In [ ]:
import requests
import re
import pandas as pd

url = "https://www.fotmob.com/es/matches/arsenal-vs-atletico-madrid/37wkxc#5205814:tab=stats"

headers = {"User-Agent": "Mozilla/5.0"}
html = requests.get(url, headers=headers).text

pattern = r'"title":"([^"]+)".{0,300}?"stats":\[(.*?),(.*?)\]'

rows = []

for m in re.finditer(pattern, html):
    stat_name = m.group(1)
    home_raw = m.group(2)
    away_raw = m.group(3)

    rows.append({
        "stat": stat_name,
        "home": home_raw,
        "away": away_raw
    })

df = pd.DataFrame(rows)

new_df = pd.concat([df.head(40), df.tail(10)], ignore_index=True)
new_df = new_df[~new_df.apply(lambda row: row.astype(str).str.contains('{').any(), axis=1)]

,stat,home,away
1,Expected goals (xG),"""1.58""","""0.53"""
2,Total shots,13,9
3,Shots on target,2,2
4,Big chances,2,1
5,Big chances missed,1,1
6,Accurate passes,"""377 (85%)""","""318 (83%)"""
7,Fouls committed,10,13
8,Corners,5,2
10,Total shots,13,9
11,Shots off target,8,4


In [37]:
arsenal_atletico_stats = new_df.copy()

arsenal_atletico_stats


,stat,home,away
1,Expected goals (xG),"""1.58""","""0.53"""
2,Total shots,13,9
3,Shots on target,2,2
4,Big chances,2,1
5,Big chances missed,1,1
6,Accurate passes,"""377 (85%)""","""318 (83%)"""
7,Fouls committed,10,13
8,Corners,5,2
10,Total shots,13,9
11,Shots off target,8,4


In [55]:
arsenal_atletico_stats

,stat,home,away
1,Expected goals (xG),"""1.58""","""0.53"""
2,Total shots,13,9
3,Shots on target,2,2
4,Big chances,2,1
5,Big chances missed,1,1
6,Accurate passes,"""377 (85%)""","""318 (83%)"""
7,Fouls committed,10,13
8,Corners,5,2
10,Total shots,13,9
11,Shots off target,8,4


In [57]:
arsenal_df = arsenal_atletico_stats.drop_duplicates(subset="stat", keep="first").reset_index(drop=True)

arsenal_df

,stat,home,away
0,Expected goals (xG),"""1.58""","""0.53"""
1,Total shots,13,9
2,Shots on target,2,2
3,Big chances,2,1
4,Big chances missed,1,1
5,Accurate passes,"""377 (85%)""","""318 (83%)"""
6,Fouls committed,10,13
7,Corners,5,2
8,Shots off target,8,4
9,Blocked shots,3,3


In [58]:
wanted_stats = [
    "Shots on target",
    "Successful dribbles",
    "Fouls committed",
    "Big chances",
    "Touches in opposition box"
]

arsenal_filtered = arsenal_df[arsenal_df["stat"].isin(wanted_stats)].set_index("stat")

arsenal_filtered

,home,away
stat,,
Shots on target,2,2
Big chances,2,1
Fouls committed,10,13
Touches in opposition box,20,20
Successful dribbles,"""3 (38%)""","""2 (20%)"""


In [59]:
arsenal_goals = 1
arsenal_fouls = int(arsenal_filtered.loc["Fouls committed"]["home"]) + int(arsenal_filtered.loc["Fouls committed"]["away"])

fouls_per_goal_arsenal = arsenal_fouls / arsenal_goals

In [60]:
new_row = pd.DataFrame({
    "home": [fouls_per_goal_arsenal],
    "away": [None]
}, index=["Fouls per goal"])

arsenal_filtered = pd.concat([arsenal_filtered, new_row])

arsenal_filtered

,home,away
Shots on target,2,2
Big chances,2,1
Fouls committed,10,13
Touches in opposition box,20,20
Successful dribbles,"""3 (38%)""","""2 (20%)"""
Fouls per goal,23.0,None


In [62]:
def extract_number(x):
    return int(str(x).split(" ")[0].replace('"', ''))

arsenal_filtered.loc["Successful dribbles", "home"] = extract_number(arsenal_filtered.loc["Successful dribbles", "home"])
arsenal_filtered.loc["Successful dribbles", "away"] = extract_number(arsenal_filtered.loc["Successful dribbles", "away"])

arsenal_filtered["home"] = pd.to_numeric(arsenal_filtered["home"], errors="ignore")
arsenal_filtered["away"] = pd.to_numeric(arsenal_filtered["away"], errors="ignore")

/var/folders/_n/0w2ts2h1483b6ccpkpr7ptg00000gn/T/ipykernel_20945/1615014280.py:7: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  arsenal_filtered["home"] = pd.to_numeric(arsenal_filtered["home"], errors="ignore")
/var/folders/_n/0w2ts2h1483b6ccpkpr7ptg00000gn/T/ipykernel_20945/1615014280.py:8: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  arsenal_filtered["away"] = pd.to_numeric(arsenal_filtered["away"], errors="ignore")


In [63]:
arsenal_filtered

,home,away
Shots on target,2.0,2.0
Big chances,2.0,1.0
Fouls committed,10.0,13.0
Touches in opposition box,20.0,20.0
Successful dribbles,3.0,2.0
Fouls per goal,23.0,NaN


In [64]:
arsenal_filtered["total"] = arsenal_filtered["home"] + arsenal_filtered["away"]

arsenal_filtered

,home,away,total
Shots on target,2.0,2.0,4.0
Big chances,2.0,1.0,3.0
Fouls committed,10.0,13.0,23.0
Touches in opposition box,20.0,20.0,40.0
Successful dribbles,3.0,2.0,5.0
Fouls per goal,23.0,NaN,NaN


In [65]:
arsenal_filtered["fouls_per_goal"] = None

arsenal_filtered.loc[:, "fouls_per_goal"] = [
    10 if idx == "Fouls committed" else None
    for idx in arsenal_filtered.index
]

In [66]:
arsenal_filtered

,home,away,total,fouls_per_goal
Shots on target,2.0,2.0,4.0,None
Big chances,2.0,1.0,3.0,None
Fouls committed,10.0,13.0,23.0,10
Touches in opposition box,20.0,20.0,40.0,None
Successful dribbles,3.0,2.0,5.0,None
Fouls per goal,23.0,NaN,NaN,None


In [67]:
arsenal_filtered.loc["Fouls committed", "fouls_per_goal"] = 23
arsenal_filtered = arsenal_filtered.drop("Fouls per goal")

In [68]:
arsenal_filtered

,home,away,total,fouls_per_goal
Shots on target,2.0,2.0,4.0,None
Big chances,2.0,1.0,3.0,None
Fouls committed,10.0,13.0,23.0,23
Touches in opposition box,20.0,20.0,40.0,None
Successful dribbles,3.0,2.0,5.0,None


In [70]:
bayern_filtered = pd.DataFrame({
    "home": [5, 2, 14, 20, 12],
    "away": [8, 6, 4, 52, 22]
}, index=[
    "Shots on target",
    "Big chances",
    "Fouls committed",
    "Touches in opposition box",
    "Successful dribbles"
])

bayern_filtered

,home,away
Shots on target,5,8
Big chances,2,6
Fouls committed,14,4
Touches in opposition box,20,52
Successful dribbles,12,22


In [71]:
bayern_filtered["total"] = bayern_filtered["home"] + bayern_filtered["away"]

In [72]:
bayern_goals = 9

bayern_fouls = bayern_filtered.loc["Fouls committed", "total"]

fouls_per_goal_bayern = bayern_fouls / bayern_goals

In [73]:
bayern_filtered["fouls_per_goal"] = None

bayern_filtered.loc["Fouls committed", "fouls_per_goal"] = round(fouls_per_goal_bayern, 2)

In [74]:
bayern_filtered

,home,away,total,fouls_per_goal
Shots on target,5,8,13,None
Big chances,2,6,8,None
Fouls committed,14,4,18,2.0
Touches in opposition box,20,52,72,None
Successful dribbles,12,22,34,None


In [75]:
arsenal_filtered

,home,away,total,fouls_per_goal
Shots on target,2.0,2.0,4.0,None
Big chances,2.0,1.0,3.0,None
Fouls committed,10.0,13.0,23.0,23
Touches in opposition box,20.0,20.0,40.0,None
Successful dribbles,3.0,2.0,5.0,None


In [76]:
arsenal_export = arsenal_filtered.copy()
arsenal_export["match"] = "Arsenal vs Atletico"

bayern_export = bayern_filtered.copy()
bayern_export["match"] = "Bayern vs PSG"

combined = pd.concat([arsenal_export, bayern_export])

combined = combined.reset_index().rename(columns={"index": "stat"})

combined = combined[
    ["match", "stat", "home", "away", "total", "fouls_per_goal"]
]

combined

,match,stat,home,away,total,fouls_per_goal
0,Arsenal vs Atletico,Shots on target,2.0,2.0,4.0,None
1,Arsenal vs Atletico,Big chances,2.0,1.0,3.0,None
2,Arsenal vs Atletico,Fouls committed,10.0,13.0,23.0,23
3,Arsenal vs Atletico,Touches in opposition box,20.0,20.0,40.0,None
4,Arsenal vs Atletico,Successful dribbles,3.0,2.0,5.0,None
5,Bayern vs PSG,Shots on target,5.0,8.0,13.0,None
6,Bayern vs PSG,Big chances,2.0,6.0,8.0,None
7,Bayern vs PSG,Fouls committed,14.0,4.0,18.0,2.0
8,Bayern vs PSG,Touches in opposition box,20.0,52.0,72.0,None
9,Bayern vs PSG,Successful dribbles,12.0,22.0,34.0,None


In [77]:
combined.to_csv("ucl_haram_ball_comparison.csv", index=False)